from torch.nn import *
import torch
def autopart(f):
    def autopart(*a, **k):
        if a:
            return f(*a, **k)
        def autopart(*a):
            return f(*a, **k)
        return autopart
    return autopart
@autopart
class reinforce:
    def __init__(s, f, n=4, it=1, lr=0.01, verbose=0):
        s.f = f
        s.it = it
        s.verbose = verbose
        s.m = Sequential(Linear(f.__code__.co_argcount, n), Hardswish(1), Linear(n, 1))
        s.o = torch.optim.Adam(s.m.parameters(), lr)
    def __call__(s, *x):
        t = s.f(*x)
        for _ in range(s.it):
            _ = s.m(torch.tensor(x, dtype=torch.float))
            functional.mse_loss(_, torch.tensor((t,), dtype=torch.float)).backward()
            s.o.step()
            s.o.zero_grad()
        if s.verbose:
            print(_.item(), t)
        return t
    def predict(s, *x):
        with torch.no_grad():
            return s.m(torch.tensor(x, dtype=torch.float)).item()
class cache:
    def __init__(s, f):
        s.f = f
        s.a = {}
    def __call__(s, *x):
        y = s.a.get(x)
        if y is None:
            y = s.a[x] = s.f(*x)
        return y
    def __getattr__(s, a):
        return getattr(s.f, a)

In [ ]:
from functools import partial
from dataclasses import dataclass


class autopart:
    def __init__(s, f):
        s.f = f

    def __call__(s, *a, **k):
        if a:
            return s.f(*a, **k)
        return partial(s.f, **k)

In [ ]:
from torch.nn import *
import torch

torch.set_default_tensor_type(torch.cuda.FloatTensor)


@autopart
@dataclass
class reinforce(dict):
    f: callable
    sr: int = 4
    tr: float = 0.03
    lr: float = 0.01
    bs: int = 0
    act: Module = Hardswish(1)
    amp: bool = 0
    n = 0

    def train(s):
        if not hasattr(s, "m"):
            s.m = Sequential(
                Linear(len(next(iter(s))), s.sr), s.act, Linear(s.sr, 1)
            ).cuda()
            s.o = torch.optim.Adam(s.m.parameters(), s.lr)
            if s.amp:
                s.s = torch.cuda.amp.GradScaler()
        x = torch.tensor(tuple(s), dtype=torch.float)
        y = torch.tensor(tuple(s.values()), dtype=torch.float)[:, None]
        for _ in range(8):
            for _ in range(int(len(s) * s.tr)):
                s.o.zero_grad()
                if s.bs:
                    _ = torch.randint(0, len(x), (s.bs,))
                _ = functional.smooth_l1_loss(
                    s.m(x[_] if s.bs else x), y[_] if s.bs else y
                )
                if s.amp:
                    s.s.scale(_).backward()
                    s.s.step(s.o)
                    s.s.update()
                else:
                    _.backward()
                    s.o.step()
            print(_)
        s.n = len(s)

    def __getitem__(s, x):
        y = s.get(x)
        if y is None:
            y = s[x] = s.f(*x)
        return y

    def __call__(s, *x):
        if s.n < len(s):
            s.train()
        with torch.no_grad():
            if len(x) < 2 and isinstance(x[0], (list, np.ndarray, torch.Tensor)):
                return s.m(torch.tensor(x[0], dtype=torch.float)).cpu()
            return s.m(torch.tensor(x, dtype=torch.float)).item()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np


@reinforce(amp=1)
def add(a, b):
    return a + b


plt.imshow([[add[i, j] for j in range(64)] for i in range(64)])

In [ ]:
plt.imshow(add(np.mgrid[:64.,:64.].transpose(1,2,0)))

In [ ]:
# type("", (Module,), {"forward": torch.sinh})()
@reinforce(sr=16, tr=500, lr=1e-3)
def mul(a, b):
    return (a * b) % 16


plt.imshow([[mul[i, j] for j in np.arange(0, 16)] for i in np.arange(0, 16)])

In [ ]:
mul.train()

In [ ]:
plt.imshow(mul(np.mgrid[:16, :16].transpose(1, 2, 0)))